In [ ]:
!git clone --recurse-submodules https://github.com/atonalfreerider/anycam.git
!cd anycam && pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!cd anycam && pip install -r requirements.txt
!pip install --upgrade moviepy

In [ ]:
%%bash
cd anycam
chmod +x download_checkpoints.sh
./download_checkpoints.sh anycam_seq8

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set path to your video (change the path below as needed)
video_path = "/content/drive/MyDrive/Zouk/3DPose/carlos-aline/spin/carlos-aline-spin-cam2.mp4"

In [ ]:
%cd /content/anycam/anycam

# Run AnyCam with varying focal length support and JSON output
!python scripts/anycam_demo.py \
    ++input_path="/content/drive/MyDrive/Zouk/3D-Pose/carlos-aline/spin/carlos-aline-spin-cam2.mp4" \
    ++model_path=../pretrained_models/anycam_seq8 \
    ++output_path=../outputs \
    ++export_json=true \
    ++ba_refinement=true

In [ ]:
# Load and examine the JSON results with varying focal lengths
import json
import numpy as np
from pathlib import Path

# Load the JSON results
results_path = Path("../outputs/camera_tracking_results.json")
with open(results_path, 'r') as f:
    results = json.load(f)

print(f"Loaded results for {results['metadata']['num_frames']} frames")
print(f"Description: {results['metadata']['description']}")

# Analyze focal length variation
focal_lengths_x = [frame['camera_intrinsics']['focal_length']['fx'] for frame in results['frames']]
focal_lengths_y = [frame['camera_intrinsics']['focal_length']['fy'] for frame in results['frames']]

print(f"\nFocal length X variation:")
print(f"  Min: {min(focal_lengths_x):.2f}")
print(f"  Max: {max(focal_lengths_x):.2f}")
print(f"  Mean: {np.mean(focal_lengths_x):.2f}")
print(f"  Std: {np.std(focal_lengths_x):.2f}")

print(f"\nFocal length Y variation:")
print(f"  Min: {min(focal_lengths_y):.2f}")
print(f"  Max: {max(focal_lengths_y):.2f}")
print(f"  Mean: {np.mean(focal_lengths_y):.2f}")
print(f"  Std: {np.std(focal_lengths_y):.2f}")

# Show sample frame data
print(f"\nSample frame data (frame 0):")
sample_frame = results['frames'][0]
print(f"  Frame ID: {sample_frame['frame_id']}")
print(f"  Focal length: fx={sample_frame['camera_intrinsics']['focal_length']['fx']:.2f}, fy={sample_frame['camera_intrinsics']['focal_length']['fy']:.2f}")
print(f"  Principal point: cx={sample_frame['camera_intrinsics']['principal_point']['cx']:.2f}, cy={sample_frame['camera_intrinsics']['principal_point']['cy']:.2f}")
print(f"  Translation: {sample_frame['camera_pose']['translation']}")

In [ ]:
# Visualize focal length changes over time
import matplotlib.pyplot as plt

frames = list(range(len(focal_lengths_x)))

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(frames, focal_lengths_x, 'b-', label='fx')
plt.plot(frames, focal_lengths_y, 'r-', label='fy')
plt.xlabel('Frame')
plt.ylabel('Focal Length (pixels)')
plt.title('Focal Length Variation Over Time')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
focal_diff = np.array(focal_lengths_x) - np.array(focal_lengths_y)
plt.plot(frames, focal_diff, 'g-')
plt.xlabel('Frame')
plt.ylabel('fx - fy (pixels)')
plt.title('Focal Length Difference (fx - fy)')
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"Focal length range: {min(focal_lengths_x):.1f} - {max(focal_lengths_x):.1f} pixels")
print(f"Total variation: {max(focal_lengths_x) - min(focal_lengths_x):.1f} pixels")